The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

In [1]:
!pip install git+https://github.com/d2l-ai/d2l-zh@release  # installing d2l


  Cloning https://github.com/d2l-ai/d2l-zh (to revision release) to /tmp/pip-req-build-7_txw5i8
  Running command git clone --filter=blob:none --quiet https://github.com/d2l-ai/d2l-zh /tmp/pip-req-build-7_txw5i8
  Running command git checkout -b release --track origin/release
  Switched to a new branch 'release'
  branch 'release' set up to track 'origin/release'.
  Resolved https://github.com/d2l-ai/d2l-zh to commit 843d3d41dca48d8df65f4b324dd171d8bfe9c067
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of d2l to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Pytho

# 数值稳定性和模型初始化
:label:`sec_numerical_stability`

到目前为止，我们实现的每个模型都是根据某个预先指定的分布来初始化模型的参数。
有人会认为初始化方案是理所当然的，忽略了如何做出这些选择的细节。甚至有人可能会觉得，初始化方案的选择并不是特别重要。
相反，初始化方案的选择在神经网络学习中起着举足轻重的作用，
它对保持数值稳定性至关重要。
此外，这些初始化方案的选择可以与非线性激活函数的选择有趣的结合在一起。
我们选择哪个函数以及如何初始化参数可以决定优化算法收敛的速度有多快。
糟糕选择可能会导致我们在训练时遇到梯度爆炸或梯度消失。
本节将更详细地探讨这些主题，并讨论一些有用的启发式方法。
这些启发式方法在整个深度学习生涯中都很有用。

## 梯度消失和梯度爆炸

考虑一个具有$L$层、输入$\mathbf{x}$和输出$\mathbf{o}$的深层网络。
每一层$l$由变换$f_l$定义，
该变换的参数为权重$\mathbf{W}^{(l)}$，
其隐藏变量是$\mathbf{h}^{(l)}$（令 $\mathbf{h}^{(0)} = \mathbf{x}$）。
我们的网络可以表示为：

$$\mathbf{h}^{(l)} = f_l (\mathbf{h}^{(l-1)}) \text{ 因此 } \mathbf{o} = f_L \circ \ldots \circ f_1(\mathbf{x}).$$

如果所有隐藏变量和输入都是向量，
我们可以将$\mathbf{o}$关于任何一组参数$\mathbf{W}^{(l)}$的梯度写为下式：

$$\partial_{\mathbf{W}^{(l)}} \mathbf{o} = \underbrace{\partial_{\mathbf{h}^{(L-1)}} \mathbf{h}^{(L)}}_{ \mathbf{M}^{(L)} \stackrel{\mathrm{def}}{=}} \cdot \ldots \cdot \underbrace{\partial_{\mathbf{h}^{(l)}} \mathbf{h}^{(l+1)}}_{ \mathbf{M}^{(l+1)} \stackrel{\mathrm{def}}{=}} \underbrace{\partial_{\mathbf{W}^{(l)}} \mathbf{h}^{(l)}}_{ \mathbf{v}^{(l)} \stackrel{\mathrm{def}}{=}}.$$

换言之，该梯度是$L-l$个矩阵
$\mathbf{M}^{(L)} \cdot \ldots \cdot \mathbf{M}^{(l+1)}$
与梯度向量 $\mathbf{v}^{(l)}$的乘积。
因此，我们容易受到数值下溢问题的影响.
当将太多的概率乘在一起时，这些问题经常会出现。
在处理概率时，一个常见的技巧是切换到对数空间，
即将数值表示的压力从尾数转移到指数。
不幸的是，上面的问题更为严重：
最初，矩阵 $\mathbf{M}^{(l)}$ 可能具有各种各样的特征值。
他们可能很小，也可能很大；
他们的乘积可能非常大，也可能非常小。

不稳定梯度带来的风险不止在于数值表示；
不稳定梯度也威胁到我们优化算法的稳定性。
我们可能面临一些问题。
要么是*梯度爆炸*（gradient exploding）问题：
参数更新过大，破坏了模型的稳定收敛；
要么是*梯度消失*（gradient vanishing）问题：
参数更新过小，在每次更新时几乎不会移动，导致模型无法学习。

### (**梯度消失**)

曾经sigmoid函数$1/(1 + \exp(-x))$（ :numref:`sec_mlp`提到过）很流行，
因为它类似于阈值函数。
由于早期的人工神经网络受到生物神经网络的启发，
神经元要么完全激活要么完全不激活（就像生物神经元）的想法很有吸引力。
然而，它却是导致梯度消失问题的一个常见的原因，
让我们仔细看看sigmoid函数为什么会导致梯度消失。


In [2]:
%matplotlib inline
import torch
from d2l import torch as d2l

x = torch.arange(-8.0, 8.0, 0.1, requires_grad=True)
y = torch.sigmoid(x)
y.backward(torch.ones_like(x))

d2l.plot(x.detach().numpy(), [y.detach().numpy(), x.grad.numpy()],
         legend=['sigmoid', 'gradient'], figsize=(4.5, 2.5))

ModuleNotFoundError: No module named 'd2l'

正如上图，当sigmoid函数的输入很大或是很小时，它的梯度都会消失。
此外，当反向传播通过许多层时，除非我们在刚刚好的地方，
这些地方sigmoid函数的输入接近于零，否则整个乘积的梯度可能会消失。
当我们的网络有很多层时，除非我们很小心，否则在某一层可能会切断梯度。
事实上，这个问题曾经困扰着深度网络的训练。
因此，更稳定的ReLU系列函数已经成为从业者的默认选择（虽然在神经科学的角度看起来不太合理）。

### [**梯度爆炸**]

相反，梯度爆炸可能同样令人烦恼。
为了更好地说明这一点，我们生成100个高斯随机矩阵，并将它们与某个初始矩阵相乘。
对于我们选择的尺度（方差$\sigma^2=1$），矩阵乘积发生爆炸。
当这种情况是由于深度网络的初始化所导致时，我们没有机会让梯度下降优化器收敛。


In [ ]:
M = torch.normal(0, 1, size=(4,4))
print('一个矩阵 \n',M)
for i in range(100):
    M = torch.mm(M,torch.normal(0, 1, size=(4, 4)))

print('乘以100个矩阵后\n', M)

一个矩阵 
 tensor([[-0.7872,  2.7090,  0.5996, -1.3191],
        [-1.8260, -0.7130, -0.5521,  0.1051],
        [ 1.1213,  1.0472, -0.3991, -0.3802],
        [ 0.5552,  0.4517, -0.3218,  0.5214]])
乘以100个矩阵后
 tensor([[-2.1897e+26,  8.8308e+26,  1.9813e+26,  1.7019e+26],
        [ 1.3110e+26, -5.2870e+26, -1.1862e+26, -1.0189e+26],
        [-1.6008e+26,  6.4559e+26,  1.4485e+26,  1.2442e+26],
        [ 3.0943e+25, -1.2479e+26, -2.7998e+25, -2.4050e+25]])


### 打破对称性

神经网络设计中的另一个问题是其参数化所固有的对称性。
假设我们有一个简单的多层感知机，它有一个隐藏层和两个隐藏单元。
在这种情况下，我们可以对第一层的权重$\mathbf{W}^{(1)}$进行重排列，
并且同样对输出层的权重进行重排列，可以获得相同的函数。
第一个隐藏单元与第二个隐藏单元没有什么特别的区别。
换句话说，我们在每一层的隐藏单元之间具有排列对称性。

假设输出层将上述两个隐藏单元的多层感知机转换为仅一个输出单元。
想象一下，如果我们将隐藏层的所有参数初始化为$\mathbf{W}^{(1)} = c$，
$c$为常量，会发生什么？
在这种情况下，在前向传播期间，两个隐藏单元采用相同的输入和参数，
产生相同的激活，该激活被送到输出单元。
在反向传播期间，根据参数$\mathbf{W}^{(1)}$对输出单元进行微分，
得到一个梯度，其元素都取相同的值。
因此，在基于梯度的迭代（例如，小批量随机梯度下降）之后，
$\mathbf{W}^{(1)}$的所有元素仍然采用相同的值。
这样的迭代永远不会打破对称性，我们可能永远也无法实现网络的表达能力。
隐藏层的行为就好像只有一个单元。
请注意，虽然小批量随机梯度下降不会打破这种对称性，但暂退法正则化可以。

## 参数初始化

解决（或至少减轻）上述问题的一种方法是进行参数初始化，
优化期间的注意和适当的正则化也可以进一步提高稳定性。

### 默认初始化

在前面的部分中，例如在 :numref:`sec_linear_concise`中，
我们使用正态分布来初始化权重值。如果我们不指定初始化方法，
框架将使用默认的随机初始化方法，对于中等难度的问题，这种方法通常很有效。

### Xavier初始化
:label:`subsec_xavier`

让我们看看某些*没有非线性*的全连接层输出（例如，隐藏变量）$o_{i}$的尺度分布。
对于该层$n_\mathrm{in}$输入$x_j$及其相关权重$w_{ij}$，输出由下式给出

$$o_{i} = \sum_{j=1}^{n_\mathrm{in}} w_{ij} x_j.$$

权重$w_{ij}$都是从同一分布中独立抽取的。
此外，让我们假设该分布具有零均值和方差$\sigma^2$。
请注意，这并不意味着分布必须是高斯的，只是均值和方差需要存在。
现在，让我们假设层$x_j$的输入也具有零均值和方差$\gamma^2$，
并且它们独立于$w_{ij}$并且彼此独立。
在这种情况下，我们可以按如下方式计算$o_i$的平均值和方差：

$$
\begin{aligned}
    E[o_i] & = \sum_{j=1}^{n_\mathrm{in}} E[w_{ij} x_j] \\&= \sum_{j=1}^{n_\mathrm{in}} E[w_{ij}] E[x_j] \\&= 0, \\
    \mathrm{Var}[o_i] & = E[o_i^2] - (E[o_i])^2 \\
        & = \sum_{j=1}^{n_\mathrm{in}} E[w^2_{ij} x^2_j] - 0 \\
        & = \sum_{j=1}^{n_\mathrm{in}} E[w^2_{ij}] E[x^2_j] \\
        & = n_\mathrm{in} \sigma^2 \gamma^2.
\end{aligned}
$$

保持方差不变的一种方法是设置$n_\mathrm{in} \sigma^2 = 1$。
现在考虑反向传播过程，我们面临着类似的问题，尽管梯度是从更靠近输出的层传播的。
使用与前向传播相同的推断，我们可以看到，除非$n_\mathrm{out} \sigma^2 = 1$，
否则梯度的方差可能会增大，其中$n_\mathrm{out}$是该层的输出的数量。
这使得我们进退两难：我们不可能同时满足这两个条件。
相反，我们只需满足：

$$
\begin{aligned}
\frac{1}{2} (n_\mathrm{in} + n_\mathrm{out}) \sigma^2 = 1 \text{ 或等价于 }
\sigma = \sqrt{\frac{2}{n_\mathrm{in} + n_\mathrm{out}}}.
\end{aligned}
$$

这就是现在标准且实用的*Xavier初始化*的基础，
它以其提出者 :cite:`Glorot.Bengio.2010` 第一作者的名字命名。
通常，Xavier初始化从均值为零，方差
$\sigma^2 = \frac{2}{n_\mathrm{in} + n_\mathrm{out}}$
的高斯分布中采样权重。
我们也可以将其改为选择从均匀分布中抽取权重时的方差。
注意均匀分布$U(-a, a)$的方差为$\frac{a^2}{3}$。
将$\frac{a^2}{3}$代入到$\sigma^2$的条件中，将得到初始化值域：

$$U\left(-\sqrt{\frac{6}{n_\mathrm{in} + n_\mathrm{out}}}, \sqrt{\frac{6}{n_\mathrm{in} + n_\mathrm{out}}}\right).$$

尽管在上述数学推理中，“不存在非线性”的假设在神经网络中很容易被违反，
但Xavier初始化方法在实践中被证明是有效的。

### 额外阅读

上面的推理仅仅触及了现代参数初始化方法的皮毛。
深度学习框架通常实现十几种不同的启发式方法。
此外，参数初始化一直是深度学习基础研究的热点领域。
其中包括专门用于参数绑定（共享）、超分辨率、序列模型和其他情况的启发式算法。
例如，Xiao等人演示了通过使用精心设计的初始化方法
 :cite:`Xiao.Bahri.Sohl-Dickstein.ea.2018`，
可以无须架构上的技巧而训练10000层神经网络的可能性。

如果有读者对该主题感兴趣，我们建议深入研究本模块的内容，
阅读提出并分析每种启发式方法的论文，然后探索有关该主题的最新出版物。
也许会偶然发现甚至发明一个聪明的想法，并为深度学习框架提供一个实现。

## 小结

* 梯度消失和梯度爆炸是深度网络中常见的问题。在参数初始化时需要非常小心，以确保梯度和参数可以得到很好的控制。
* 需要用启发式的初始化方法来确保初始梯度既不太大也不太小。
* ReLU激活函数缓解了梯度消失问题，这样可以加速收敛。
* 随机初始化是保证在进行优化前打破对称性的关键。
* Xavier初始化表明，对于每一层，输出的方差不受输入数量的影响，任何梯度的方差不受输出数量的影响。

## 练习

1. 除了多层感知机的排列对称性之外，还能设计出其他神经网络可能会表现出对称性且需要被打破的情况吗？
2. 我们是否可以将线性回归或softmax回归中的所有权重参数初始化为相同的值？
3. 在相关资料中查找两个矩阵乘积特征值的解析界。这对确保梯度条件合适有什么启示？
4. 如果我们知道某些项是发散的，我们能在事后修正吗？看看关于按层自适应速率缩放的论文 :cite:`You.Gitman.Ginsburg.2017` 。


[Discussions](https://discuss.d2l.ai/t/1818)


其他神经网络中需要打破的对称性除了多层感知机

（MLP）隐藏单元之间的置换对称性，许多架构也存在类似或更广义的对称性：

卷积神经网络（CNN）的卷积核冗余：如果在同一通道组内将多个不同的卷积核（filter）初始化为完全相同的权重，它们在相同感受野上提取的特征图和接收到的反向传播梯度完全一致，卷积核将永远学出相同特征，退化为单个卷积核。

Multi-Head Attention（多头自注意力机制）：

多头机制的本意是让模型在不同的投影子空间中捕获不同位置、不同类型的相关性。如果将各头的投影矩阵 $W_i^Q, W_i^K, W_i^V$ 初始化为相同的值，各注意力头将产生完全相同的注意力权重与输出，失去多头多视角的表达能力。循环神经网络（RNN/LSTM/GRU）中的门控与隐状态：多层 RNN 同一层内的多个隐单元如果初始化完全一致，同样无法解耦状态表征。此外，多维隐藏状态的转移矩阵行/列对称也会导致维度塌陷。

混合专家模型（MoE, Mixture of Experts）：

多个结构相同的专家网络如果以相同参数初始化，门控网络（Router）对它们输出相同得分，且每个专家反向传播更新完全一致，无法自发分工专业化。

线性回归与 Softmax 回归全初始化为相同值线性回归（凸优化，单输出）：

可行。线性回归的目标函数是严格凸函数（只要特征矩阵满列秩）。若将所有特征的权重初始化为同一个常数（如全 0 或全 $c$），优化算法（如梯度下降）依然能够正常工作。因为每个参数 $w_j$ 对应的输入特征 $x_j$ 数值通常不同，梯度 $\frac{\partial L}{\partial w_j} = -(y - \hat{y})x_j$ 天然不同，在第一次反向传播迭代后对称性就会立即被打破。Softmax 回归（多类分类）：可以初始化为全 0，但有退化细节需注意：若将权重矩阵 $W \in \mathbb{R}^{C \times D}$ 全置为 0，初始时刻各类的预测得分（logits）全为 0，softmax 预测概率分布为均匀分布（$1/C$）。由于损失函数是关于输出类别的交叉熵，对于某个样本（真实标签为 $y$），其对第 $k$ 类的梯度为 $(p_k - \mathbf{1}_{k=y}) x_j$。由于真实类别与非真实类别的误差项不同，各类别权重的梯度不同，因此类别维度的对称性也会在第一步更新时立即被打破。例外：若某两个类别在所有样本中的先验和损失反馈永远对称，或在缺乏输入区分度的无监督场景下，可能会退化。但在标准有监督凸优化设定下，全相同初始化不会锁死模型。

矩阵乘积特征值的解析界与梯度条件的启示

在深层网络的反向传播中，第 $l$ 层的梯度可表示为一系列雅可比矩阵（Jacobian）的连乘：$$J = \prod_{k=1}^{L} W_k D_k$$其中 $W_k$ 为权重矩阵，$D_k$ 为激活函数的导数对角阵。根据矩阵特征值界限理论（如 Ostrowski 矩阵乘积界、奇异值与特征值的 Weyl 不等式，以及 Gelfand 谱半径公式 $\rho(A) = \lim_{k\to\infty} \Vert{}A^k\Vert{}^{1/k}$）：奇异值乘积界：$$\sigma_{\min}(A)\sigma_{\min}(B) \le \sigma_i(AB) \le \sigma_{\max}(A)\sigma_{\max}(B)$$对于谱半径（最大特征值模长），有 $\rho(AB) \le \Vert{}AB\Vert{} \le \Vert{}A\Vert{}_2 \Vert{}B\Vert{}_2 = \sigma_{\max}(A)\sigma_{\max}(B)$。对梯度条件的启示：梯度爆炸与消失的边界控制：如果每一层的权重雅可比矩阵的最大奇异值持续大于 1（$\sigma_{\max}(W_k) > 1$），则连乘后特征值模长与范数随深度呈指数级发散（梯度爆炸）；反之，若 $\sigma_{\max}(W_k) < 1$，梯度将指数衰减归零（梯度消失）。等距映射（Isometry）原则：为了保持深层网络的梯度条件稳定（即雅可比矩阵的条件数 $\kappa(J) = \sigma_{\max}/\sigma_{\min} \approx 1$），初始化的权重矩阵最好接近正交矩阵（Orthogonal Initialization），使得矩阵的奇异值分布严格聚集在 1 附近，这也是 Xavier/He 初始化的方差缩放理论基础。

事后修正发散项：

LARS（按层自适应速率缩放）机制是的，可以在事后进行动态控制与修正。在 You, Gitman, Ginsburg (2017) 的论文《Large Batch Training of Convolutional Networks》中，作者针对超大 Batch Size 训练时出现的梯度发散与网络失稳问题，提出了 LARS (Layer-wise Adaptive Rate Scaling)：发散根源分析：在深层网络中，不同层的权重范数与梯度范数的比例差异巨大。特别是在浅层或靠近输入/输出的特定层，有时会出现梯度的 $L_2$ 范数 $\Vert{}\nabla L(W^l)\Vert{}$ 显著大于权重本身范数 $\Vert{}W^l\Vert{}$ 的情况。在大的全局学习率 $\eta$ 下，单步更新量 $\Vert{}\Delta W^l\Vert{} \gg \Vert{}W^l\Vert{}$，直接导致该层权重剧烈震荡、发散。LARS 的事后修正公式：LARS 不去逐元素缩放，而是在每层计算完梯度后，根据该层的范数比值动态调整该层的局部学习率（Local Learning Rate）：$$\lambda^l = \eta \cdot \frac{\Vert{}W^l\Vert{}}{\Vert{}\nabla L(W^l)\Vert{} + \beta \Vert{}W^l\Vert{}}$$其中 $\eta$ 是全局学习率，$\beta$ 是权重衰减系数。实际用于更新的层学习率被严格限制在该层权重的相对尺度内。结论：即便某些层的反向传播梯度在数值上出现过大或发散倾向，LARS 通过按层范数自适应缩放，强制将每一步的更新步长锚定在当前层权重的承受范围内，从而稳定了大批次、高学习率下的训练轨迹。